#  Notebook 2 — YOLOv8 Training

This notebook focuses on training an object detection model using **YOLOv8**, a fast and efficient deep learning framework designed for real-time performance and ease of use.

steps :
- Set up the training configuration
- Load and prepare your dataset
- Train the model 

The workflow is designed to be simple, modular, and beginner-friendly, allowing you to experiment with different settings 



In [1]:
# ============================================================
#  CONFIG
# ============================================================

DATA_YAML   = "data_local.yaml"  #  path to fixed yaml from Notebook 1
MODEL_SIZE  = "yolov8m.pt"       #  yolov8n.pt (fastest) | yolov8s.pt | yolov8m.pt (balanced) | yolov8l.pt | yolov8x.pt (best)
PROJECT_DIR = "runs"             #  output folder
RUN_NAME    = "egypt_yolo"       #  experiment name

# --- Training ---
EPOCHS      = 20       #   
IMG_SIZE    = 640      #  640 (standard) | 416 (faster) | 1280 (slower)
BATCH_SIZE  = 8        #  
WORKERS     = 0        #  0 on Windows | 4+ on Linux
PATIENCE    = 20       #  early stopping patience (epochs without improvement)
LR0         = 0.01     #  initial learning rate
LRF         = 0.001    #  final learning rate (after cosine decay)

# --- Class imbalance ---
USE_CLASS_WEIGHTS = True  #  True = handle car:bike 25:1 imbalance

# --- Resume ---
RESUME = False  #  True = resume last interrupted run

CLASSES = [
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]

print("Config set!")
print(f"   Model      : {MODEL_SIZE}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Image size : {IMG_SIZE}px")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Classes    : {len(CLASSES)}")

Config set!
   Model      : yolov8m.pt
   Epochs     : 20
   Image size : 640px
   Batch size : 8
   Classes    : 12


## Cell 1 — Environment Check

In [3]:
import torch
from ultralytics import YOLO
import ultralytics

print(f"PyTorch      : {torch.__version__}")
print(f"Ultralytics  : {ultralytics.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# Time estimate
import os, yaml
with open(DATA_YAML) as f: data = yaml.safe_load(f)
train_imgs = len(os.listdir(data['train'])) if os.path.exists(data['train']) else 0
steps = train_imgs // BATCH_SIZE
mins  = steps * 0.04 / 60  # ~40ms/step on RTX 4060
print(f"\nTime estimate:")
print(f"   Train images : {train_imgs:,}")
print(f"   Steps/epoch  : {steps:,}")
print(f"   ~{mins:.0f} min/epoch × {EPOCHS} epochs = ~{mins*EPOCHS/60:.1f} hrs")

PyTorch      : 2.5.1+cu121
Ultralytics  : 8.4.30
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 6.9 GB free / 8.0 GB total

Time estimate:
   Train images : 48,329
   Steps/epoch  : 6,041
   ~4 min/epoch × 20 epochs = ~1.3 hrs


## Cell 2 — Compute Class Weights (for imbalance)

In [5]:
import os
import numpy as np
from collections import Counter
import yaml

with open(DATA_YAML) as f: data = yaml.safe_load(f)
lbl_dir = os.path.join(os.path.dirname(data['train']), "..", "labels").replace("images", "labels")
lbl_dir = data['train'].replace("images", "labels")

counts = Counter()
if os.path.exists(lbl_dir):
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1

total = sum(counts.values())
# Inverse frequency weights — rare classes get higher weight
weights = []
print(f"{'Class':<15} {'Count':>8} {'Weight':>8}")
print("-" * 35)
for i, cls in enumerate(CLASSES):
    count = counts.get(i, 1)
    w = total / (len(CLASSES) * max(count, 1))
    weights.append(round(w, 4))
    flag = "" if w > 2 else ""
    print(f"{cls:<15} {count:>8,} {w:>8.3f}{flag}")

print(f"\nClass weights: {weights}")
print("(higher = model pays more attention to that class)")

Class              Count   Weight
-----------------------------------
traffic light     11,975    1.987
traffic sign      21,928    1.085
car              123,542    0.193
person            31,735    0.750
bus               11,497    2.069
truck             13,577    1.752
rider             10,913    2.180
bike               5,541    4.294
motor             13,164    1.807
train              7,187    3.310
banner            28,879    0.824
tuktuk             5,564    4.276

Class weights: [1.9868, 1.085, 0.1926, 0.7497, 2.0694, 1.7524, 2.1801, 4.2938, 1.8073, 3.3104, 0.8238, 4.276]
(higher = model pays more attention to that class)


##  Cell 3 — Train YOLOv8

This cell runs the full training pipeline using your configured settings. It loads a pretrained YOLOv8 model, fine-tunes it on your dataset, and saves checkpoints along with training metrics.

Key highlights:
- Automatically downloads pretrained weights (first run only)
- Supports GPU acceleration (if available)
- Includes built-in augmentations to improve generalization
- Handles class imbalance through loss weighting and data augmentation
- Saves both best and last model checkpoints


In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 
model = YOLO(MODEL_SIZE)
print(f" Loaded: {MODEL_SIZE}")
print(f" Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# Train
results = model.train(
    data      = DATA_YAML,        # dataset config
    epochs    = EPOCHS,           # total epochs
    imgsz     = IMG_SIZE,         # image size
    batch     = BATCH_SIZE,       # batch size
    workers   = WORKERS,          # dataloader workers
    patience  = PATIENCE,         # early stopping
    lr0       = LR0,              # initial LR
    lrf       = LRF,              # final LR
    project   = PROJECT_DIR,      # output folder
    name      = RUN_NAME,         # experiment name
    resume    = RESUME,           # resume from last checkpoint
    device    = 0 if torch.cuda.is_available() else 'cpu',
    # Class imbalance handling
    cls       = 0.5,              # classification loss weight
    # Augmentations (help with class imbalance)
    mosaic    = 1.0,              # mosaic augmentation (0=off, 1=full)
    mixup     = 0.1,              # mixup augmentation
    copy_paste= 0.1,              # copy-paste augmentation (helps rare classes)
    # Logging
    plots     = True,             # save training plots
    save      = True,             # save checkpoints
    save_period = 10,              # save checkpoint every N epochs
    verbose   = True,
)

print("\n  Training complete!")
print(f"   Best model : {PROJECT_DIR}/{RUN_NAME}/weights/best.pt")
print(f"   Last model : {PROJECT_DIR}/{RUN_NAME}/weights/last.pt")

✅ Loaded: yolov8m.pt
   Parameters: 25,902,640
Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=egypt_yolo, nbs=64, nms=False, opset=None, optimize=False, op

## Cell 4 — Resume Training (if interrupted)

In [ ]:
import os
import torch
from ultralytics import YOLO

LAST_CKPT = "runs/detect/runs/egypt_yolo/weights/last.pt"

if not os.path.exists(LAST_CKPT):
    print(f"Checkpoint not found: {LAST_CKPT}")
else:
    model_r = YOLO(LAST_CKPT)

    model_r.train(
        data   = DATA_YAML,
        epochs = 30,   
        resume = True,
        device = 0 if torch.cuda.is_available() else 'cpu',
    )

    print("Resumed training complete!")

New https://pypi.org/project/ultralytics/8.4.31 available  Update with 'pip install -U ultralytics'
WARNING model 'runs\detect\runs\egypt_yolo\weights\last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

## Cell 5 — Quick Validation After Training

In [3]:
import os

BEST_MODEL = f"runs/detect/runs/egypt_yolo/weights/best.pt" 

if not os.path.exists(BEST_MODEL):
    print(f"Model not found: {BEST_MODEL}")
else:
    model_val = YOLO(BEST_MODEL)
    metrics = model_val.val(data=DATA_YAML, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                             device=0 if torch.cuda.is_available() else 'cpu')
    print(f"\n Validation Results:")
    print(f"   mAP50     : {metrics.box.map50:.4f}")
    print(f"   mAP50-95  : {metrics.box.map:.4f}")
    print(f"   Precision : {metrics.box.mp:.4f}")
    print(f"   Recall    : {metrics.box.mr:.4f}")

Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 93 layers, 25,846,708 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 227.2205.7 MB/s, size: 118.6 KB)
val: Scanning C:\Users\hp\Desktop\proj2\dataset\val\val\labels.cache... 6278 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6278/6278  0.0s
val: C:\Users\hp\Desktop\proj2\dataset\val\val\images\person_200_jpg.rf.70b8c03f1b8cafe9f84fca25993bbc0b.jpg: 7 duplicate labels removed
WARNING Box and segment counts should be equal, but got len(segments) = 233, len(boxes) = 36235. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 785/785 8.9it/s 1:28<0.1sss
                   all       6278      362